> # Mobile Game In-App Purchases Analysis

The dataset contains information about in-app purchases made by users in a mobile game. The goal of this analysis is to understand the spending behavior of users and identify any patterns or trends that can help improve the game's monetization strategy.

# Exploratory Data Analysis (EDA)

In [ ]:
!pip install scikit-learn-extra
import numpy as np
import pandas as pd
import random
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots
from sklearn.preprocessing import TargetEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import plotly.express as px
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as sch
from sklearn_extra.cluster import KMedoids
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score # for determine the optimal number of clusters.. ( Hierarchical Clustering )


data = "/content/mobile_game_inapp_purchases.csv"
df = pd.read_csv(data)
df.info()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

ImportError: numpy.core.multiarray failed to import (auto-generated because you didn't call 'numpy.import_array()' after cimporting numpy; use '<void>numpy._import_array' to disable if you are certain you don't need it).

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

The dataset contains some missing values in the 'InAppPurchaseAmount' column, which is the target variable for our analysis. We will need to handle these missing values before proceeding with any further analysis.

In [ ]:
numeric_features = df.drop('InAppPurchaseAmount',axis=1).select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_features = df.select_dtypes(include=['object','bool']).columns.tolist()
encoder = TargetEncoder()
encoded_data = df.copy()
encoded_data = encoded_data.dropna(subset='InAppPurchaseAmount')
for col in categorical_features:
    encoded_data[col] = encoder.fit_transform(encoded_data[[col]], encoded_data['InAppPurchaseAmount'])

In [ ]:
plt.figure(figsize = (10,10))
sns.heatmap(encoded_data.corr(), annot=True, cmap='coolwarm')

From the correlation heatmap, we can see that there are some features that have a strong correlation with the target variable 'InAppPurchaseAmount' which is _'SpendingSegment'_ have a positive correlation with _'InAppPurchaseAmount'_, while _'Age'_ has a negative correlation. This suggests that younger users are less likely to spend money on the game.

In [ ]:
fig = make_subplots(rows=2, cols=len(numeric_features))
for i in range(1,len(numeric_features)+1):
    fig.add_trace(go.Histogram(x=df[numeric_features[i-1]], name=f'{numeric_features[i-1]} Distribution'), row=1, col=i)
    fig.add_trace(go.Scatter(x=df[numeric_features[i-1]], y=df['InAppPurchaseAmount'], mode='markers', name=f'{numeric_features[i-1]} vs InAppPurchaseAmount'), row=2, col=i)
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=df['InAppPurchaseAmount'], name='InAppPurchaseAmount Distribution'))

The distribution of 'InAppPurchaseAmount' is highly skewed, with a large number of users making small purchases and a few users making large purchases. This is a common pattern in mobile games, where a small percentage of users (often referred to as "whales") account for a significant portion of the revenue.

In [ ]:
country_purchase_mean = df.groupby("Country")["InAppPurchaseAmount"].mean().sort_values(ascending=False)
fig = go.Figure()
fig.add_trace(go.Bar(x= country_purchase_mean.index,y=country_purchase_mean))

 From the bar chart, we can see that there are some countries where users tend to spend more on in-app purchases compared to others. This information can be useful for targeting marketing efforts and optimizing the game's monetization strategy in different regions.

In [ ]:
genre_purchase_mean = df.groupby("GameGenre")["InAppPurchaseAmount"].mean().sort_values(ascending=False)
fig = go.Figure()
fig.add_trace(go.Bar(x= genre_purchase_mean.index,y=genre_purchase_mean))

The bar chart shows that certain game genres tend to have higher average in-app purchase amounts compared to others. This information can be useful for game developers to focus on creating content and features that are more likely to encourage spending among users.

In [ ]:
method_purchase_mean = df.groupby("PaymentMethod")["InAppPurchaseAmount"].mean().sort_values(ascending=False)
fig = go.Figure()
fig.add_trace(go.Bar(x=method_purchase_mean.index,y=method_purchase_mean))

The bar chart indicates that certain payment methods are associated with higher average in-app purchase amounts. This information can be valuable for game developers and marketers to optimize the payment options available to users and potentially increase revenue.

In [ ]:
spending_segments = df['SpendingSegment'].value_counts().sort_index()
spending_segments_mean = df.groupby("SpendingSegment")["InAppPurchaseAmount"].mean().sort_index()
fig = go.Figure()
fig.add_trace(go.Bar(x=spending_segments.index, y=spending_segments.values, name='Number of users in each spending segment'))
fig.add_trace(go.Bar(x=spending_segments_mean.index, y=spending_segments_mean.values, name='Amount of money spent by the segment'))


The bar chart shows the distribution of users across different spending segments and the average amount of money spent by each segment. This information can help game developers understand the spending behavior of their user base and tailor their monetization strategies accordingly.

In [ ]:
dicti = df['Gender'].value_counts().to_dict()
px.pie(values = df['Gender'].value_counts(),names=df['Gender'].value_counts().index)

The pie chart shows the distribution of users by gender. This information can be useful for game developers and marketers to understand the demographics of their user base and create targeted marketing campaigns or game content that appeals to specific

In [ ]:
dicti = df['Device'].value_counts().to_dict()
px.pie(values = df['Device'].value_counts(),names=df['Device'].value_counts().index)

The pie chart shows the distribution of users by device type. This information can be valuable for game developers to optimize the game's performance and user experience on different devices, as well as for marketers to target their advertising efforts based on the most popular device types among their user base.

# Data Preprocessing

In [ ]:
df.head()

Since each row represents a user, we use `UserID` as the index to make the dataset more structured and easier to work with

In [ ]:
df.drop("UserID", axis=1, inplace=True)

We divide the dataset into numerical and categorical features for easier preprocessing

In [ ]:
num_cols = df.select_dtypes(np.number).columns
cat_cols = df.select_dtypes('object').columns

define a preprocessing function that imputes missing **numerical** values using the **median** *(robust to outliers)* and **categorical** values using the **most frequent** category.

In [ ]:
def handle_missing_values(df):
    num_imputer = SimpleImputer(strategy='median')
    df[num_cols] = num_imputer.fit_transform(df[num_cols])

    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

    return df

In [ ]:
df = handle_missing_values(df)
df.isna().sum()

All missing values have been handled,

plot **boxplots** for **numerical** columns to understand their distribution and **detect outliers**.

In [ ]:
plt.figure(figsize=(15, 10))

for i, col in enumerate(num_cols):
    plt.subplot(3, 2, i + 1)
    sns.boxplot(x=df[col])

plt.tight_layout()
plt.show()

implementing an outlier treatment function that uses the **IQR** technique to **cap extreme values instead of removing them**, preserving data size

In [ ]:
def filter_outliers(df, cols):
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        df[col] = np.where(df[col] < lower, lower, df[col])
        df[col] = np.where(df[col] > upper, upper, df[col])
    return df

Evaluating the effect of outlier handling

In [ ]:
df = filter_outliers(df, ['SessionCount',"InAppPurchaseAmount"])
df[['SessionCount',"InAppPurchaseAmount"]].plot(kind="box")
plt.show()

The distributions appear more controlled after outlier treatment

checking **how many unique values** exist in each **categorical** column

In [ ]:
df.select_dtypes(include='object').nunique()

replacing "Other" values with the **most frequent category** in the *Gender* column

In [ ]:
df.replace("Other", df["Gender"].mode()[0], inplace=True)

Applying **one-hot encoding** for **low-cardinality** features and **frequency encoding** for **high-cardinality** features like *Country*

In [ ]:
def encode_features(df):
    onehot_cols = ["Gender", "Device", "SpendingSegment", "PaymentMethod", "GameGenre"]
    df = pd.get_dummies(df, columns=onehot_cols, drop_first=True)

    freq = df["Country"].value_counts() / len(df)
    df["Country"] = df["Country"].map(freq)

    return df

In [ ]:
df = encode_features(df)

transforming the "LastPurchaseDate" into a more useful numerical feature by calculating the number of days since the last purchase

In [ ]:
def transform_last_purchase(df):
    df["LastPurchaseDate"] = pd.to_datetime(df["LastPurchaseDate"])
    df["DaysSinceLastPurchase"] = (df["LastPurchaseDate"].max() - df["LastPurchaseDate"]).dt.days

    df = df.drop(columns=["LastPurchaseDate"])

    return df

In [ ]:
df = transform_last_purchase(df)
df["DaysSinceLastPurchase"].head()

We inspect the "DaysSinceLastPurchase" column to ensure the transformation was applied correctly

Standardization all features so they have similar ranges

In [ ]:
def scale_dataframe(df):
    scaler = StandardScaler()

    matrix_scaled = scaler.fit_transform(df)
    df_scaled = pd.DataFrame(matrix_scaled,columns=df.columns,index=df.index)

    return df_scaled

df = scale_dataframe(df)

In [ ]:
df.head()

The **dataset** is now **clean** and ready for modeling

# Dimensionality Reduction

# PCA
This Dimensionality reduction technique will not be used for the main clustering model because interpretability is more important than compression.
eg. K-Medoids selects actual data points as centers (medoids). After PCA, these data points are transformed into a new, abstract coordinate system (principal components)

In [ ]:
try:
  import ipyvolume as ipv
except:
  !pip install ipyvolume -q
  import ipyvolume as ipv
  !jupyter nbextension enable --py --sys-prefix ipyvolume

def visualize_3d(x, labels):
    # Workaround as axis limits are not auto-scaling
    x_norm = (x - x.min(axis=0)) / (x.max(axis=0) - x.min(axis=0))
    fig = ipv.figure(height=400, width=400)
    x, y, z = x_norm[:, 0], x_norm[:, 1], x_norm[:, 2]

    # Colors
    cmap = plt.get_cmap('Greens', 3)
    color = cmap(labels)
    ipv.scatter(x, y, z, size=4, marker="sphere", color=color)
    ipv.show()

In [ ]:
from sklearn.decomposition import PCA

X = df.drop(columns=['InAppPurchaseAmount'])
y = df['InAppPurchaseAmount']

pca = PCA(n_components=3)
df_pca = pca.fit_transform(X)

print("Original shape:", df.shape)
print("Reduced shape:", df_pca.shape)

In [ ]:
visualize_3d(df_pca, y)

In [ ]:
df_pca = pd.DataFrame(df_pca, columns=["PC1", "PC2", "PC3"])
df_pca.head()

# Genetic Algorithm (Feature selection, which is one type of dimensionality reduction.)

In [ ]:
POP_SIZE = 20
N_GENERATIONS = 30
MUTATION_RATE = 0.1
N_CLUSTERS = 3

def fitness(X, chromosome):
    if np.sum(chromosome) == 0:
        return -1

    X_selected = X[:, chromosome == 1]
    try:
        model = KMedoids(n_clusters=N_CLUSTERS)
        labels = model.fit_predict(X_selected)
        score = silhouette_score(X_selected, labels)
        penalty = 0.01 * np.sum(chromosome)
        return score - penalty
    except:
        return -1

def initialize_population(n_features):
    return np.random.randint(0, 2, (POP_SIZE, n_features)) #(20, 31) shape

def select(population, fitnesses):
    i, j = random.sample(range(len(population)), 2)
    return population[i] if fitnesses[i] > fitnesses[j] else population[j]

def crossover(parent1, parent2):
    point = random.randint(1, len(parent1) - 1)
    child = np.concatenate([parent1[:point], parent2[point:]]) # First part from parent1, second from parent2
    return child

def mutate(chromosome):
    for i in range(len(chromosome)):
        if random.random() < MUTATION_RATE:
            chromosome[i] = 1 - chromosome[i]
    return chromosome

def genetic_feature_selection(X):
    n_features = X.shape[1]
    population = initialize_population(n_features)

    best_solution = None
    best_score = -1

    for gen in range(N_GENERATIONS):
        fitnesses = np.array([fitness(X, ind) for ind in population])

        max_idx = np.argmax(fitnesses)
        if fitnesses[max_idx] > best_score:
            best_score = fitnesses[max_idx]
            best_solution = population[max_idx].copy()

        new_population = []

        for _ in range(POP_SIZE):
            parent1 = select(population, fitnesses)
            parent2 = select(population, fitnesses)

            child = crossover(parent1, parent2)
            child = mutate(child)

            new_population.append(child)

        population = np.array(new_population)

        print(f"Generation {gen+1}, Best Score: {best_score:.4f}")

    return best_solution

X = df.copy()
X_np = X.values

best_features = genetic_feature_selection(X_np)

# Apply best features
X_reduced = X_np[:, best_features == 1]

print("Selected features:", np.where(best_features == 1)[0])
print("Reduced shape:", X_reduced.shape)

# ========================

# Task 3: K-Medoid Clustring

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

In [ ]:
# 1. Calculating the Silhouette Score for different values of k using only Manhattan distance
best_score = -1
best_k = -1
scores = []
K_range = range(2, 11)

for k in K_range:
# Using the Manhattan metric directly
    model = KMedoids(n_clusters=k, metric='manhattan', init='random', random_state=42)
    labels = model.fit_predict(scaled_data ) # 'data' is the data after Scaling

    score = silhouette_score(scaled_data, labels)
    scores.append(score)

    if score > best_score:
        best_score = score
        best_k = k

# Drawing the Silhouette Score
plt.figure(figsize=(10, 5))
plt.plot(K_range, scores, marker='o', color='green', label='Manhattan Metric')
plt.title('K-Medoids: Silhouette Score vs Number of Clusters (K)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 2. Applying the Elbow Method (Inertia) for confirmation
inertia = []
for k in range(1, 11):
    model = KMedoids(n_clusters=k, metric='manhattan', init='random', random_state=42)
    model.fit(scaled_data)
    inertia.append(model.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(range(1, 11), inertia, marker='o', color='purple')
plt.title('Elbow Method for K-Medoids (Manhattan)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Total Cost)')
plt.grid(True)
plt.show()

In [ ]:
# 3. the final result
print(f"Optimal number of clusters (K): {best_k}")
print(f"Best Silhouette Score with Manhattan: {best_score:.4f}")

# Building the final model using the optimal K
Kmedoids_model = KMedoids(n_clusters=best_k, metric='manhattan', init='random', random_state=42)
Kmedoids_model.fit(scaled_data )

In [ ]:
labels= Kmedoids_model.labels_
print(labels)

In [ ]:
centers=Kmedoids_model.cluster_centers_
print("centers", centers)

In [ ]:
print(Kmedoids_model.medoid_indices_)

In [ ]:
for j in range(1,k+1):
  print("\nCluster", j)
  print(scaled_data[labels == j-1])

In [ ]:
plt.scatter(scaled_data[:,0], scaled_data[:,1], c=Kmedoids_model.labels_, cmap='rainbow')
plt.title('Clusters of points')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

# ============ KMedoids After DR ============

In [ ]:
# 1. Calculating the Silhouette Score for different values of k using only Manhattan distance
best_score = -1
best_k = -1
scores = []
K_range = range(2, 11)

for k in K_range:
# Using the Manhattan metric directly
    model = KMedoids(n_clusters=k, metric='manhattan', init='random', random_state=42)
    labels = model.fit_predict(X_reduced) # 'data' is the data after Scaling

    score = silhouette_score(X_reduced, labels)
    scores.append(score)

    if score > best_score:
        best_score = score
        best_k = k

# Drawing the Silhouette Score
plt.figure(figsize=(10, 5))
plt.plot(K_range, scores, marker='o', color='green', label='Manhattan Metric')
plt.title('K-Medoids: Silhouette Score vs Number of Clusters (K)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 2. Applying the Elbow Method (Inertia) for confirmation
inertia = []
for k in range(1, 11):
    model = KMedoids(n_clusters=k, metric='manhattan', init='random', random_state=42)
    model.fit(X_reduced)
    inertia.append(model.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(range(1, 11), inertia, marker='o', color='purple')
plt.title('Elbow Method for K-Medoids (Manhattan)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Total Cost)')
plt.grid(True)
plt.show()

In [ ]:
# 3. the final result
print(f"Optimal number of clusters (K): {best_k}")
print(f"Best Silhouette Score with Manhattan: {best_score:.4f}")

# Building the final model using the optimal K
Kmedoids_model = KMedoids(n_clusters=best_k, metric='manhattan', init='random', random_state=42)
Kmedoids_model.fit(X_reduced )

In [ ]:
labels= Kmedoids_model.labels_
print(labels)

In [ ]:
centers=Kmedoids_model.cluster_centers_
print("centers", centers)

In [ ]:
print(Kmedoids_model.medoid_indices_)

In [ ]:
for j in range(1,k+1):
  print("\nCluster", j)
  print(X_reduced[labels == j-1])

In [ ]:
plt.scatter(X_reduced[:,0], X_reduced[:,1], c=Kmedoids_model.labels_, cmap='rainbow')
plt.title('Clusters of points')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

# ========================
# ========================

# Task 4: Hierarical Clustring



In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)

In [ ]:
# Determine the optimal number of clusters with linkage_methods using the Silhouette Score

linkage_methods = ['single', 'complete', 'average']
best_score = -1
best_k = -1
best_linkage = ''

plt.figure(figsize=(12, 6))

for linkage in linkage_methods:
    scores = []

    for k in range(2, 11):
        cluster = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage=linkage)
        labels = cluster.fit_predict(scaled_data)

        score = silhouette_score(scaled_data, labels)
        scores.append(score)

        if score > best_score:
            best_score = score
            best_k = k
            best_linkage = linkage

    plt.plot(range(2, 11), scores, marker='o', label=linkage)

plt.title('Silhouette Score vs Number of Clusters (K)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Elbow Method for Hierarchical Clustering

Z = sch.linkage(scaled_data, method=best_linkage)
distances = Z[-15:, 2]

plt.figure(figsize=(12, 6))

plt.plot(range(1, len(distances) + 1), distances[::-1], marker='o')
plt.title('Elbow Method (Merge Distances)')
plt.xlabel('Number of Clusters')
plt.ylabel('Distance (Jump in merge)')
plt.grid(True)
plt.show()

In [ ]:
# The best data inputs

print(f"Optimal number of clusters: {best_k}")
print(f"Best Linkage Method: {best_linkage}")
print(f"Best Silhouette Score: {best_score:.4f}")

In [ ]:
# According to the best data inputs (Hierarchical Clustering Dendrogram)

plt.figure(figsize=(12, 6))
plt.title(f'Hierarchical Clustering Dendrogram ({best_linkage} linkage)')
dendrogram = sch.dendrogram(sch.linkage(scaled_data, method=best_linkage), truncate_mode="lastp", p=100)
plt.xlabel('Cluster Size / Index')
plt.ylabel('Distance')
plt.show()

In [ ]:
 # Analyze and interpret each cluster profile in terms of the domain context

final_hc = AgglomerativeClustering(n_clusters=best_k, metric='euclidean', linkage=best_linkage)

df_curr_copy = df.copy()
df_curr_copy['Cluster'] = final_hc.fit_predict(scaled_data)

cluster_profiles = df_curr_copy.groupby('Cluster').mean()

print("\nCluster Profiles (Mean values):")
display(cluster_profiles)

### Cluster Profiling & Domain Context Analysis

Based on the Hierarchical Clustering results, the algorithm optimally divided the player base into **2 distinct clusters**. By analyzing the mean values of the features within each cluster, we can deeply interpret the users' behavioral profiles in the context of mobile gaming monetization:

#### **Cluster 0: The "Free-to-Play" Majority (Casuals & Minnows)**
* **Data Characteristics:** * This group shows below-average spending, indicated by a negative `InAppPurchaseAmount` (-0.043) and a negative `SpendingSegment_Whale` feature (-0.151).
  * They score higher on the `SpendingSegment_Minnow` feature (0.052), meaning if they do spend, it is in very minimal amounts.
  * Their `SessionCount` and `AverageSessionLength` are stable and close to the overall dataset average.
* **Domain Interpretation:** This cluster represents the massive base of standard users who rely almost entirely on the free-to-play mechanics. They enjoy the game but avoid spending real money.
* **Actionable Strategy:** Since direct revenue from this group is extremely low, the business strategy should focus on **Ad-based Monetization** (e.g., rewarded video ads for extra lives/currency) and offering highly discounted "Starter Packs" to break the spending barrier and convert them into paying users.

#### **Cluster 1: The "Whales" (High-Value VIP Players)**
* **Data Characteristics:**
  * This group exhibits a massive positive spike in `InAppPurchaseAmount` (1.897) and an exceptionally high `SpendingSegment_Whale` indicator (6.593).
  * They log in more frequently, shown by a higher `SessionCount` (0.274).
  * Interestingly, their `AverageSessionLength` is shorter than average (-0.249), suggesting they might log in quickly to make purchases, open loot boxes, or participate in VIP events rather than grinding for hours.
* **Domain Interpretation:** These are the core VIP players ("Whales"). Although they are a smaller percentage of the total user base, they generate the vast majority of the game's actual revenue.
* **Actionable Strategy:** Retaining this cluster is the highest priority. The company should assign dedicated **VIP Customer Support** for these users, introduce exclusive high-tier cosmetic items, and avoid interrupting their gameplay with ads. Loyalty programs and premium battle passes will keep this segment highly engaged.

# ============ Hierarchical Clustering After DR ============

In [ ]:
# Determine the optimal number of clusters with linkage_methods using the Silhouette Score

linkage_methods = ['single', 'complete', 'average']
best_score = -1
best_k = -1
best_linkage = ''

plt.figure(figsize=(12, 6))

for linkage in linkage_methods:
    scores = []

    for k in range(2, 11):
        cluster = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage=linkage)
        labels = cluster.fit_predict(X_reduced)

        score = silhouette_score(X_reduced, labels)
        scores.append(score)

        if score > best_score:
            best_score = score
            best_k = k
            best_linkage = linkage

    plt.plot(range(2, 11), scores, marker='o', label=linkage)

plt.title('Silhouette Score vs Number of Clusters (K)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Elbow Method for Hierarchical Clustering

Z = sch.linkage(X_reduced, method=best_linkage)
distances = Z[-15:, 2]

plt.figure(figsize=(12, 6))

plt.plot(range(1, len(distances) + 1), distances[::-1], marker='o')
plt.title('Elbow Method (Merge Distances)')
plt.xlabel('Number of Clusters')
plt.ylabel('Distance (Jump in merge)')
plt.grid(True)
plt.show()

In [ ]:
# The best data inputs

print(f"Optimal number of clusters: {best_k}")
print(f"Best Linkage Method: {best_linkage}")
print(f"Best Silhouette Score: {best_score:.4f}")

In [ ]:
# According to the best data inputs (Hierarchical Clustering Dendrogram)

plt.figure(figsize=(12, 6))
plt.title(f'Hierarchical Clustering Dendrogram ({best_linkage} linkage)')
dendrogram = sch.dendrogram(sch.linkage(X_reduced, method=best_linkage), truncate_mode="lastp", p=100)
plt.xlabel('Cluster Size / Index')
plt.ylabel('Distance')
plt.show()

### Assigning New Data Points to Agglomerative Clusters (Workaround)

Since `AgglomerativeClustering` does not have a `predict` method for new data, we can implement a workaround by:
1.  Calculating the centroids (mean) of the existing clusters.
2.  For a new data point, finding the closest centroid and assigning it to that cluster.

In [ ]:
# 1. First, re-run AgglomerativeClustering to get labels for the original `X_reduced` data
hc_model_for_prediction = AgglomerativeClustering(n_clusters=best_k, metric='euclidean', linkage=best_linkage)
original_data_labels = hc_model_for_prediction.fit_predict(X_reduced)

# 2. Calculate the centroids for each cluster

# 3. Define a function to assign a new (scaled and feature-reduced) data point to the closest cluster
def assign_new_point_to_cluster(new_point_scaled_reduced, centroids):
    distances = [np.linalg.norm(new_point_scaled_reduced - centroid) for centroid in centroids]
    closest_cluster_id = np.argmin(distances)
    return closest_cluster_id


# ============================== FUZZY ============================

In [ ]:
!pip install scikit-fuzzy

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

cluster_type = ctrl.Antecedent(np.arange(0, 2, 1),'cluster_type')
session_count = ctrl.Antecedent(np.arange(0, 31, 1), 'session_count')
session_length = ctrl.Antecedent(np.arange(0, 61, 1), 'session_length')
days_to_first_purchase = ctrl.Antecedent(np.arange(0, 91, 1), 'days_to_first_purchase')
purchase = ctrl.Antecedent(np.arange(0, 5001, 1), 'purchase')
behavior = ctrl.Consequent(np.arange(0, 101, 1), 'behavior')

#-----------------------------------MARKOV Was Here-------------------------------#

cluster_type['free_to_play'] = fuzz.trimf(cluster_type.universe, [-0.5, 0, 0.5])
cluster_type['whale'] = fuzz.trimf(cluster_type.universe,[0.5, 1, 1.5])

session_count['low'] = fuzz.trimf(session_count.universe, [0, 0, 5])
session_count['medium'] = fuzz.trimf(session_count.universe, [4, 10, 18])
session_count['high'] = fuzz.trimf(session_count.universe, [15, 30, 30])

session_length['short'] = fuzz.trimf(session_length.universe, [0, 0, 10])
session_length['medium'] = fuzz.trimf(session_length.universe, [8, 18, 30])
session_length['long'] = fuzz.trimf(session_length.universe, [25, 45, 60])

days_to_first_purchase['fast'] = fuzz.trimf(days_to_first_purchase.universe, [0, 0, 7])
days_to_first_purchase['normal'] = fuzz.trimf(days_to_first_purchase.universe, [5, 18, 35])
days_to_first_purchase['late'] = fuzz.trimf(days_to_first_purchase.universe, [30, 90, 90])

purchase['low'] = fuzz.trimf(purchase.universe, [0, 0, 300])
purchase['medium'] = fuzz.trimf(purchase.universe, [250, 900, 2000])
purchase['high'] = fuzz.trimf(purchase.universe, [1500, 5000, 5000])

behavior['low'] = fuzz.trimf(behavior.universe, [0, 0, 35])
behavior['medium'] = fuzz.trimf(behavior.universe, [30, 55, 75])
behavior['high'] = fuzz.trimf(behavior.universe, [70, 100, 100])

In [ ]:
cluster_type.view()
session_count.view()
session_length.view()
days_to_first_purchase.view()
purchase.view()
behavior.view()

In [ ]:
ct_levels = {
    'free_to_play': cluster_type['free_to_play'],
    'whale': cluster_type['whale']
}

sc_levels = {
    'low': session_count['low'],
    'medium': session_count['medium'],
    'high': session_count['high']
}

sl_levels = {
    'short': session_length['short'],
    'medium': session_length['medium'],
    'long': session_length['long']
}

dtfp_levels = {
    'fast': days_to_first_purchase['fast'],
    'normal': days_to_first_purchase['normal'],
    'late': days_to_first_purchase['late']
}

purchase_levels = {
    'low': purchase['low'],
    'medium': purchase['medium'],
    'high': purchase['high']
}

In [ ]:
def determine_behavior(ct, sc, sl, dtfp, p):

    # ---- WHALE CLUSTER ----
    if ct == 'whale':
        if p == 'high':
            return behavior['high']

        if p == 'medium':
            if sc in ['medium', 'high']:
                return behavior['high']
            return behavior['medium']

        if sc == 'high' and sl in ['medium', 'long']:
            return behavior['medium']
        return behavior['low']

    # ---- FREE-TO-PLAY CLUSTER ----
    if ct == 'free_to_play':
        if p == 'high':
            return behavior['high']

        if p == 'medium':
            if dtfp == 'fast':
                return behavior['high']
            return behavior['medium']

        if sc == 'high' and sl == 'long':
            return behavior['medium']

        if dtfp == 'fast' and sc in ['medium', 'high']:
            return behavior['medium']

        return behavior['low']

In [ ]:
rules = []
rule_id = 1
for ct_name, ct_term in ct_levels.items():
  for sc_name, sc_term in sc_levels.items():
      for sl_name, sl_term in sl_levels.items():
          for dtfp_name, dtfp_term in dtfp_levels.items():
              for p_name, p_term in purchase_levels.items():

                  output = determine_behavior(
                      ct_name, sc_name, sl_name, dtfp_name, p_name
                  )

                  rule = ctrl.Rule(
                      ct_term &
                      sc_term &
                      sl_term &
                      dtfp_term &
                      p_term,
                      output
                  )

                  rules.append(rule)
                  rule_id += 1

In [ ]:
CRM_system = ctrl.ControlSystem(rules)
CRM_simulator = ctrl.ControlSystemSimulation(CRM_system)

In [ ]:
#test
CRM_simulator.input['cluster_type'] = 0
CRM_simulator.input['session_count'] = 5
CRM_simulator.input['session_length'] = 10
CRM_simulator.input['days_to_first_purchase'] = 10
CRM_simulator.input['purchase'] = 2000
CRM_simulator.compute()
value = CRM_simulator.output['behavior']
print(f"Player Monetization Score is {value:.2f}%")

In [ ]:
session_count.view(CRM_simulator)
session_length.view(CRM_simulator)
days_to_first_purchase.view(CRM_simulator)
purchase.view(CRM_simulator)

In [ ]:
behavior.view(CRM_simulator)

# System Implementation

In [ ]:
random_row = df.sample(n=1, random_state=42)
random_row

In [ ]:
def implementation(scaled_input_row_df, GeneticBestFeatures, ClusterModel, FuzzySimulator, scaler_obj):

    # 1. Inverse transform the scaled row to get unscaled values for Fuzzy
    # The scaler returns a NumPy array, so we recreate a DataFrame to retain column names.
    unscaled_input_values_np = scaler_obj.inverse_transform(scaled_input_row_df)
    unscaled_input_row_df = pd.DataFrame(unscaled_input_values_np, columns=scaled_input_row_df.columns, index=scaled_input_row_df.index)

    # 2. Prepare scaled features for the clustering model
    selected_scaled_features_np = scaled_input_row_df.values[:, GeneticBestFeatures == 1]

    # 3. Predict the cluster using the provided KMedoids model
    assigned_cluster = ClusterModel.predict(selected_scaled_features_np)[0]

    # 4. Populate Fuzzy inputs with correct (unscaled) values and the predicted cluster
    FuzzySimulator.input['cluster_type'] = assigned_cluster
    FuzzySimulator.input['session_count'] = unscaled_input_row_df['SessionCount'].values[0]
    FuzzySimulator.input['session_length'] = unscaled_input_row_df['AverageSessionLength'].values[0]
    FuzzySimulator.input['days_to_first_purchase'] = unscaled_input_row_df['FirstPurchaseDaysAfterInstall'].values[0]
    FuzzySimulator.input['purchase'] = unscaled_input_row_df['InAppPurchaseAmount'].values[0]

    FuzzySimulator.compute()
    value = FuzzySimulator.output['behavior']
    print(f"Player Monetization Score is {value:.2f}%")

# Call the implementation function with the corrected arguments
implementation(scaled_input_row_df=random_row, GeneticBestFeatures=best_features, ClusterModel=Kmedoids_model, FuzzySimulator=CRM_simulator, scaler_obj=scaler)

## Conclusion

This analysis shows that in-app purchase behavior is highly uneven: most users spend little, while a small group contributes a large share of revenue. Spending also varies meaningfully across user segments, countries, game genres, and payment methods, which suggests monetization performance is strongly influenced by player profile and market context rather than a single global pattern. The correlation analysis further indicates that `SpendingSegment` is strongly associated with `InAppPurchaseAmount`, while `Age` shows a negative relationship with purchase amount in this dataset.

From a business perspective, the findings support a segmented monetization strategy. Priorities should include: (1) retention and value optimization for high-spending segments, (2) targeted offers by country and genre, and (3) streamlined promotion of high-performing payment methods to reduce purchase friction. Since missing values exist in `InAppPurchaseAmount` and this notebook focuses on EDA, the next step is to validate these patterns with statistical testing and/or predictive modeling before making high-impact product decisions.